In [2]:
import pandas as pd
import glob
import os
#import matplotlib.pyplot as plt


# ============================================================
# 1. LOAD ALL CSV FILES
# ============================================================

files = glob.glob("*.csv")

dfs = {}

for file in files:
    name = os.path.splitext(file)[0]
    dfs[name] = pd.read_csv(file)

orders_df = dfs["olist_orders_dataset"]


## Missing Value Analysis

The first step in assessing the data quality is accessing nature of missing values in the dataset.

Missing values by themselves are not a proof of a data quality issue. They occur due to many reasons such as:

- **MCAR (Missing Completely At Random):** missingness is unrelated to observed or unobserved data.
- **MAR (Missing At Random):** missingness is related to other observed variables.
- **MNAR (Missing Not At Random):** missingness is related to the value of the variable that is missing, or to factors not observed in the dataset.

These categories cannot be established from missing-value counts alone, although they provide a framework for investigating the observed missingness patterns.



In [3]:
# ============================================================
# 2. INITIAL MISSING VALUES SUMMARY
# ============================================================

missing_summary = pd.DataFrame({
    "Missing_Count": orders_df.isna().sum(),
    "Missing_Percentage": (orders_df.isna().mean() * 100).round(2),
    "Non_Missing_Count": orders_df.notna().sum()
})

missing_summary = missing_summary.sort_values(
    by="Missing_Count",
    ascending=False
)

print("INITIAL MISSING VALUES SUMMARY")
print("=" * 80)

display(missing_summary)


INITIAL MISSING VALUES SUMMARY


,Missing_Count,Missing_Percentage,Non_Missing_Count
order_delivered_customer_date,2965,2.98,96476
order_delivered_carrier_date,1783,1.79,97658
order_approved_at,160,0.16,99281
order_id,0,0.00,99441
order_purchase_timestamp,0,0.00,99441
order_status,0,0.00,99441
customer_id,0,0.00,99441
order_estimated_delivery_date,0,0.00,99441


## Investigating the Reasons for Missing Values

After identifying missing values, the next step is to investigate whether their occurrence is completely random or related with other variables in the dataset.

The timestamps in this dataset correspond to different stages of the order fulfillment lifecycle. The recorded `order_status` may therefore help explain why particular timestamps are missing indicating timestamps that may be structurally absent because an order did not reach the future stages.


In [4]:
# ============================================================
# 3. DEFINE TIMESTAMP COLUMNS
# ============================================================

date_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]


# ============================================================
# 4. MISSINGNESS BY ORDER STATUS
# ============================================================

for column in date_columns:

    print(f"\n{'=' * 80}")
    print(f"MISSINGNESS BY ORDER STATUS: {column}")
    print(f"{'=' * 80}")

    missingness_table = pd.crosstab(
        orders_df["order_status"],
        orders_df[column].isna()
    )

    # Ensure both columns exist, even if one has zero observations
    missingness_table = missingness_table.reindex(
        columns=[False, True],
        fill_value=0
    )

    missingness_table["Total"] = missingness_table.sum(axis=1)

    missingness_table.loc["All"] = missingness_table.sum(axis=0)

    missingness_table.columns = [
        "Not_Missing",
        "Missing",
        "Total"
    ]

    display(missingness_table)


MISSINGNESS BY ORDER STATUS: order_approved_at


,Not_Missing,Missing,Total
order_status,,,
approved,2,0,2
canceled,484,141,625
created,0,5,5
delivered,96464,14,96478
invoiced,314,0,314
processing,301,0,301
shipped,1107,0,1107
unavailable,609,0,609
All,99281,160,99441



MISSINGNESS BY ORDER STATUS: order_delivered_carrier_date


,Not_Missing,Missing,Total
order_status,,,
approved,0,2,2
canceled,75,550,625
created,0,5,5
delivered,96476,2,96478
invoiced,0,314,314
processing,0,301,301
shipped,1107,0,1107
unavailable,0,609,609
All,97658,1783,99441



MISSINGNESS BY ORDER STATUS: order_delivered_customer_date


,Not_Missing,Missing,Total
order_status,,,
approved,0,2,2
canceled,6,619,625
created,0,5,5
delivered,96470,8,96478
invoiced,0,314,314
processing,0,301,301
shipped,0,1107,1107
unavailable,0,609,609
All,96476,2965,99441


## Order Lifecycle and Missing Timestamps

Before analysing the results from the previous cell, let's understand the various stages involved in order processing.

A simplified order lifecycle can be represented as:

`Created` → `Approved` → `Processing` → `Invoiced` → `Shipped` → `Delivered`

Not every order necessarily progresses through every stage.

For example:

- An order may be cancelled before approval.
- An order may be approved but subsequently cancelled.
- An order may be approved but later marked as unavailable.

Therefore, the 141 cancelled orders where `order_approved_at` is missing may be logically valid if these orders were cancelled after being created but before being approved.

Similarly, the 314 orders with the status `invoiced` but missing `order_delivered_carrier_date` may be expected, since these orders have not yet progressed to the stage where they are handed over to the carrier.

These examples demonstrate that missing timestamps are not necessarily data quality issues. Their validity depends on whether the order progressed far enough in its lifecycle for the corresponding timestamp to be recorded.

## Investigating Missing Timestamps Within Each Order Status

From the previous analysis, we observed that missing timestamps are associated with different order statuses. However, we have only examined the missingness of each timestamp individually.

For example, we identified 141 cancelled orders where `order_approved_at` is missing. If these orders were cancelled before being approved, we would also expect their subsequent timestamps, such as `order_delivered_carrier_date` and `order_delivered_customer_date`, to be missing.

Similarly, orders with the status `invoiced` have 314 missing values in `order_delivered_carrier_date`. Since these orders have not yet progressed to the shipping stage, the absence of subsequent delivery timestamps would be expected.

Therefore, we will examine combinations of missing timestamps within each order status to check whether the missing values are consistent with the order's progression through the lifecycle.

This will help us identify missingness patterns that appear reasonable, as well as records where the combination of missing and non-missing timestamps may require further investigation.

In [6]:
# ============================================================
# MISSINGNESS PATTERNS FOR EACH ORDER STATUS
# ============================================================

pattern_columns = [
    "Approval_Timestamp_Missing",
    "Carrier_Timestamp_Missing",
    "Customer_Delivery_Timestamp_Missing"
]

for status in orders_df["order_status"].dropna().unique():

    status_orders = orders_df[
        orders_df["order_status"] == status
    ]

    missingness_patterns = (
        status_orders[date_columns]
        .isna()
        .value_counts()
        .reset_index(name="Count")
    )

    missingness_patterns.columns = pattern_columns + ["Count"]

    # Convert Boolean indicators into readable labels
    for column in pattern_columns:
        missingness_patterns[column] = (
            missingness_patterns[column]
            .map({True: "Yes", False: "No"})
        )

    print(f"\nMISSINGNESS PATTERNS: {status.upper()}")
    print("=" * 80)

    display(missingness_patterns)



MISSINGNESS PATTERNS: DELIVERED


,Approval_Timestamp_Missing,Carrier_Timestamp_Missing,Customer_Delivery_Timestamp_Missing,Count
0,No,No,No,96455
1,Yes,No,No,14
2,No,No,Yes,7
3,No,Yes,No,1
4,No,Yes,Yes,1



MISSINGNESS PATTERNS: INVOICED


,Approval_Timestamp_Missing,Carrier_Timestamp_Missing,Customer_Delivery_Timestamp_Missing,Count
0,No,Yes,Yes,314



MISSINGNESS PATTERNS: SHIPPED


,Approval_Timestamp_Missing,Carrier_Timestamp_Missing,Customer_Delivery_Timestamp_Missing,Count
0,No,No,Yes,1107


... (6 outputs left out)


MISSINGNESS PATTERNS: CREATED


,Approval_Timestamp_Missing,Carrier_Timestamp_Missing,Customer_Delivery_Timestamp_Missing,Count
0,Yes,Yes,Yes,5



MISSINGNESS PATTERNS: APPROVED


,Approval_Timestamp_Missing,Carrier_Timestamp_Missing,Customer_Delivery_Timestamp_Missing,Count
0,No,Yes,Yes,2


## Potentially Problematic Missingness Patterns

Based on the missingness patterns observed above, certain combinations appear inconsistent with the expected order lifecycle and may indicate data quality issues.

- **Delivered Orders:** Out of 96,478 delivered orders, 23 have at least one missing timestamp. Since these orders have reached the delivery stage, we would generally expect all three timestamps (`order_approved_at`, `order_delivered_carrier_date`, and `order_delivered_customer_date`) to be present. These records may therefore indicate data quality issues.

- **Cancelled Orders:** There are 6 cancelled orders where none of the three timestamps are missing. In particular, these records have a non-missing `order_delivered_customer_date`, despite their status being `canceled`. Since cancelled orders would generally not be expected to reach the customer delivery stage, these records appear unusual and require further investigation.

These records will be isolated for further examination to determine whether the missing timestamps or recorded order statuses are inconsistent with the expected order lifecycle.


## Timestamp Chronology Validation

In addition to missingness, the chronological order of the timestamps need to be examined.

Under the expected order fulfillment process, the following sequence 
would generally be expected:

`order_purchase_timestamp`
    ≤ `order_approved_at`
    ≤ `order_delivered_carrier_date`
    ≤ `order_delivered_customer_date`

These conditions are not necessarily applicable to every order because 
some orders may be cancelled or may not progress through every stage.

The validation is therefore performed only on records where the 
relevant timestamps are available.

In [7]:
timestamp_order_violations = orders_df[
    (
        orders_df["order_approved_at"].notna() &
        (orders_df["order_approved_at"] < orders_df["order_purchase_timestamp"])
    ) |
    (
        orders_df["order_delivered_carrier_date"].notna() &
        orders_df["order_approved_at"].notna() &
        (
            orders_df["order_delivered_carrier_date"] <
            orders_df["order_approved_at"]
        )
    ) |
    (
        orders_df["order_delivered_customer_date"].notna() &
        orders_df["order_delivered_carrier_date"].notna() &
        (
            orders_df["order_delivered_customer_date"] <
            orders_df["order_delivered_carrier_date"]
        )
    )
]

print("Number of chronological violations:", 
      len(timestamp_order_violations))

print(
    timestamp_order_violations[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date"
        ]
    ].to_string(index=False)
)

Number of chronological violations: 1382
                        order_id order_status order_purchase_timestamp   order_approved_at order_delivered_carrier_date order_delivered_customer_date
dcb36b511fcac050b97cd5c05de84dc3    delivered      2018-06-07 19:03:12 2018-06-12 23:31:02          2018-06-11 14:54:00           2018-06-21 15:34:32
688052146432ef8253587b930b01a06d    delivered      2018-04-22 08:48:13 2018-04-24 18:25:22          2018-04-23 19:19:14           2018-04-24 19:31:58
58d4c4747ee059eeeb865b349b41f53a    delivered      2018-07-21 12:49:32 2018-07-26 23:31:53          2018-07-24 12:57:00           2018-07-25 23:58:19
412fccb2b44a99b36714bca3fef8ad7b    delivered      2018-07-22 22:30:05 2018-07-23 12:31:53          2018-07-23 12:24:00           2018-07-24 19:26:42
56a4ac10a4a8f2ba7693523bb439eede    delivered      2018-07-22 13:04:47 2018-07-27 23:31:09          2018-07-24 14:03:00           2018-07-28 00:05:39
32e4fa9bb468884309b58b9348de70c3    delivered      2018-07-

## Conclusion

The missingness analysis revealed that missing timestamps are not necessarily data quality issues, as their presence or absence may depend on how far an order progressed through the fulfillment lifecycle.

However, certain missingness patterns were identified that may require further investigation:

- **Delivered Orders:** Out of 96,478 delivered orders, 23 have at least one missing timestamp. Since these orders have reached the delivery stage, we would generally expect all three timestamps (`order_approved_at`, `order_delivered_carrier_date`, and `order_delivered_customer_date`) to be present. These records may therefore indicate data quality issues.

- **Cancelled Orders:** There are 6 cancelled orders where none of the three timestamps are missing. In particular, these records have a non-missing `order_delivered_customer_date`, despite their status being `canceled`. Since cancelled orders would generally not be expected to reach the customer delivery stage, these records appear unusual and require further investigation.

In addition to missingness, timestamp chronology validation identified **1,382 records** where the available timestamps did not follow the expected order fulfillment sequence.

For example, some records contain an `order_delivered_carrier_date` that occurs before `order_approved_at`, which is inconsistent with the expected order processing sequence.

Overall, the analysis identified potential data quality issues involving missing timestamps, unexpected combinations of timestamps and order statuses, and chronological inconsistencies. These records warrant further investigation to determine whether they reflect incorrect data recording, inconsistencies in order statuses, or other underlying issues.